# Slow Enumeration of Active Directory - K-Means Clustering

## Overview

This notebook uses K-Means clustering to detect slow enumeration of Active Directory through LDAP queries. The approach analyzes Directory Services logs to identify anomalous user behavior patterns.

**Objective:** Detect users performing unusual LDAP query patterns that may indicate reconnaissance or enumeration activities.

**Data Requirements:**
- Directory Services logs with LDAP queries captured
- Event ID 1644 (contains LDAP queries)
- Fields: `event_code`, `winlog_user_name`, `winlog_event_data_param2`, `event_created`

**Expected Outputs:**
- List of anomalous user-week pairs
- Cluster analysis showing which queries define anomaly patterns
- Confusion matrix and classification metrics

**Model Approach:**
- TF-IDF feature extraction on weekly LDAP query patterns
- K-Means clustering (k=4)
- Smallest cluster identified as anomaly
- Persistent threats flagged if user appears in anomaly cluster multiple weeks

**Prerequisites:**
- MinIO/S3 access configured in environment variables
- Spark cluster accessible
- Directory Services logs available

**Notes:**
- Developed in test environment; tune hyperparameters for your dataset
- Can be adapted for other data sources (Defender for Identity, Sentinel)
- Works best with sufficient baseline data (multiple weeks of activity)

In [ ]:
# Cell 1: Imports & Environment Setup
# =========================================

import os
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import HashingTF, IDF, VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
import numpy as np

load_dotenv()

print("✅ Environment loaded")

In [ ]:
# Cell 2: Spark Configuration
# =========================================

SPARK_APP_NAME = "ADEnumeration_KMeans"

SPARK_MASTER = os.getenv('SPARK_MASTER')
SPARK_DRIVER_HOST = os.getenv('SPARK_DRIVER_HOST')
SPARK_DRIVER_PORT = int(os.getenv('SPARK_DRIVER_PORT', '7771'))
SPARK_BLOCK_MANAGER_PORT = int(os.getenv('SPARK_BLOCK_MANAGER_PORT', '7772'))
SPARK_UI_PORT = int(os.getenv('SPARK_UI_PORT', '8088'))
SPARK_EXECUTOR_CORES = int(os.getenv('SPARK_EXECUTOR_CORES', '4'))
SPARK_EXECUTOR_MEMORY = os.getenv('SPARK_EXECUTOR_MEMORY', '6G')
SPARK_DRIVER_MEMORY = os.getenv('SPARK_DRIVER_MEMORY', '2g')
SPARK_JARS_PATH = os.getenv('SPARK_JARS_PATH', '/home/jmi/Spark_cluster/master/jars/*')
SPARK_EXECUTOR_CLASSPATH = os.getenv('SPARK_EXECUTOR_CLASSPATH', '/opt/spark/jars/*:/opt/spark/external-jars/*')

print(f"✅ Spark Config: master={SPARK_MASTER}, cores={SPARK_EXECUTOR_CORES}, memory={SPARK_EXECUTOR_MEMORY}")

In [ ]:
# Cell 3: MinIO Configuration
# =========================================

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
MINIO_PATH_STYLE_ACCESS = os.getenv('MINIO_PATH_STYLE_ACCESS', 'true')

STORAGE_OPTIONS = {
    'key': MINIO_ACCESS_KEY,
    'secret': MINIO_SECRET_KEY,
    'endpoint_url': MINIO_ENDPOINT
}

print(f"✅ MinIO Config: endpoint={MINIO_ENDPOINT}")

In [ ]:
# Cell 4: Data Paths
# =========================================

DATA_PATH = os.getenv('DATA_PATH', 's3a://winlogbeat/winlogbeat/*.parquet')
BASE_S3_PATH = os.getenv('BASE_S3_PATH', 's3a://temp/anomaly_detection')

print(f"✅ Data Path: {DATA_PATH}")

In [ ]:
# Cell 5: AD Enumeration - K-Means Parameters
# =========================================

AD_KMEANS_K = int(os.getenv('AD_KMEANS_K', '4'))
AD_KMEANS_SEED = int(os.getenv('AD_KMEANS_SEED', '42'))
AD_TFIDF_NUM_FEATURES = int(os.getenv('AD_TFIDF_NUM_FEATURES', '2000'))
AD_WEEK_BUCKET_SIZE_DAYS = int(os.getenv('AD_WEEK_BUCKET_SIZE_DAYS', '7'))

print(f"✅ AD K-Means Config: k={AD_KMEANS_K}, features={AD_TFIDF_NUM_FEATURES}, week_size={AD_WEEK_BUCKET_SIZE_DAYS} days")

In [ ]:
# Cell 6: Create Spark Session
# =========================================

spark = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .master(SPARK_MASTER)

    .config("spark.driver.host", SPARK_DRIVER_HOST)
    .config("spark.driver.port", str(SPARK_DRIVER_PORT))
    .config("spark.blockManager.port", str(SPARK_BLOCK_MANAGER_PORT))
    .config("spark.ui.port", str(SPARK_UI_PORT))

    .config("spark.executor.cores", str(SPARK_EXECUTOR_CORES))
    .config("spark.executor.memory", SPARK_EXECUTOR_MEMORY)
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY)

    .config("spark.executor.extraClassPath", SPARK_EXECUTOR_CLASSPATH)
    .config("spark.jars", SPARK_JARS_PATH)
    .config("spark.executor.userClassPathFirst", "true")

    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", MINIO_PATH_STYLE_ACCESS)

    .getOrCreate()
)

print(f"✅ Spark session created: {SPARK_APP_NAME}")
print(f"   Master: {SPARK_MASTER}")

In [ ]:
# Cell 7: Load Data
# =========================================

df_raw = spark.read.parquet(DATA_PATH)
print(f"✅ Loaded {df_raw.count()} records from MinIO")

## Data Filtering

The following code:
1. Adds a human-readable time column from epoch time
2. Filters data based on requirements (event code 1644, non-null usernames and queries)
3. Filters out ANONYMOUS LOGON and SYSTEM users (LDAP queries require authenticated network users)

**Key Fields:**
- `event_code`: Event ID from Directory Services logs
- `winlog_user_name`: Username who initiated LDAP query
- `winlog_event_data_param2`: LDAP query name
- `event_created`: Event creation time in epoch
- `readable_time`: Event creation time in human-readable format

In [ ]:
# Cell 8: Filter Data
# =========================================

df_with_time = df_raw.withColumn(
    "readable_time",
    F.from_unixtime(F.col("`event_created`") / 1000000000)
)

df_cleaned = (
    df_with_time
    .where(F.col("event_code") == "1644")
    .where(
        F.col("event_code").isNotNull() &
        F.col("winlog_user_name").isNotNull() &
        F.col("winlog_event_data_param2").isNotNull()
    )
    .where(~F.col("winlog_user_name").isin(["ANONYMOUS LOGON", "SYSTEM"]))
    .select(
        F.col("event_code"),
        F.col("winlog_user_name"),
        F.col("winlog_event_data_param2"),
        F.col("event_created"),
        F.col("readable_time")
    )
)

print(f"✅ Filtered to {df_cleaned.count()} records (event_code=1644, non-null users/queries)")

df_cleaned.printSchema()
df_cleaned.show(5, truncate=False)

## Feature Engineering

The next step engineers features from the data:

**Steps:**
1. **Week Buckets:** Assign a week number to every event based on days since the earliest date
2. **User-Week Aggregation:** Create a document for each user's activity in a specific week
3. **TF-IDF:** Transform query lists into numerical features
   - HashingTF: Creates fixed-size vector (numFeatures=2000)
   - IDF: Rescales features based on global frequency (rare queries = high scores)
4. **Feature Assembly:** Combine TF-IDF with volume and variety metrics
5. **Standardization:** Scale features for K-Means

In [ ]:
# Cell 9: Feature Engineering
# =========================================

min_date_obj = df_cleaned.agg(F.min("readable_time")).collect()[0][0]

df_weekly = df_cleaned.withColumn(
    "days_since_start",
    F.datediff(F.col("readable_time"), F.lit(min_date_obj))
).withColumn(
    "week_id",
    F.floor(F.col("days_since_start") / AD_WEEK_BUCKET_SIZE_DAYS)
)

print(f"✅ Created weekly buckets (size={AD_WEEK_BUCKET_SIZE_DAYS} days)")

user_weekly_history = df_weekly.groupBy("winlog_user_name", "week_id").agg(
    F.collect_list("winlog_event_data_param2").alias("weekly_queries"),
    F.count("*").alias("week_volume"),
    F.approx_count_distinct("winlog_event_data_param2").alias("week_variety")
)

hashingTF = HashingTF(inputCol="weekly_queries", outputCol="raw_features", numFeatures=AD_TFIDF_NUM_FEATURES)
idf = IDF(inputCol="raw_features", outputCol="tfidf_features")

assembler = VectorAssembler(
    inputCols=["tfidf_features", "week_volume", "week_variety"],
    outputCol="features_vec"
)

scaler = StandardScaler(inputCol="features_vec", outputCol="features")

pipeline = Pipeline(stages=[hashingTF, idf, assembler, scaler])
model = pipeline.fit(user_weekly_history)
df_processed = model.transform(user_weekly_history)

print(f"✅ Feature engineering complete ({AD_TFIDF_NUM_FEATURES} features)")

## Clustering & Analysis

**Approach:**
1. **K-Means Clustering:** Group user-weeks into k clusters
2. **Anomaly Identification:** Smallest cluster is identified as anomaly (typically unusual patterns)
3. **Persistent Threats:** Flag users appearing in anomaly cluster for multiple weeks
4. **Explainability:** Analyze cluster center to understand which queries define anomalies

In [ ]:
# Cell 10: K-Means Clustering
# =========================================

kmeans = KMeans(k=AD_KMEANS_K, seed=AD_KMEANS_SEED)
km_model = kmeans.fit(df_processed)
df_predictions = km_model.transform(df_processed)

cluster_counts = df_predictions.groupBy("prediction").count().orderBy("count")
cluster_counts.show()

anomaly_cluster_id = cluster_counts.first()[0]

df_anomalies = df_predictions.filter(F.col("prediction") == anomaly_cluster_id)

persistent_threats = df_anomalies.groupBy("winlog_user_name").agg(
    F.count("*").alias("anomalous_week_count"),
    F.collect_list("week_id").alias("weeks_anomalous")
).orderBy(F.desc("anomalous_week_count"))

print(f"✅ Clustering complete (k={AD_KMEANS_K})")
print(f"   Anomaly cluster ID: {anomaly_cluster_id}")

print("\nUsers who appeared in anomaly clusters frequently:")
persistent_threats.show(10, truncate=False)

In [ ]:
# Cell 11: Cluster Explainability
# =========================================

centers = km_model.clusterCenters()
anomaly_center = centers[anomaly_cluster_id]

top_indices = np.argsort(anomaly_center[:AD_TFIDF_NUM_FEATURES])[-10:]

print(f"\nTop 10 Hash Indices defining Anomaly Cluster:")
for idx in top_indices:
    print(f"Index: {idx}, Weight: {anomaly_center[idx]:.4f}")

print("\n--- Most frequent queries in Anomaly Cluster ---")
df_anomalies_exploded = df_anomalies.select("weekly_queries").withColumn("query", F.explode("weekly_queries"))
top_anomaly_queries = df_anomalies_exploded.groupBy("query").count().orderBy(F.desc("count"))
top_anomaly_queries.show(10, truncate=False)

print("\n--- Most frequent queries globally ---")
df_global_exploded = user_weekly_history.select("weekly_queries").withColumn("query", F.explode("weekly_queries"))
top_global_queries = df_global_exploded.groupBy("query").count().orderBy(F.desc("count"))
top_global_queries.show(10, truncate=False)

## Results Analysis

The following code joins anomalous user-week pairs back to raw data to retrieve timestamps and query details for analysis.

In [ ]:
# Cell 12: Analyze Anomalous Events
# =========================================

anomalous_keys = df_predictions.filter(F.col("prediction") == anomaly_cluster_id).select(
    "winlog_user_name",
    "week_id"
)

print("Anomalous (User, Week) pairs:")
anomalous_keys.show(10, truncate=False)

df_anomalous_events = df_weekly.join(
    anomalous_keys,
    on=["winlog_user_name", "week_id"],
    how="inner"
)

print(f"\nAnomalous events: {df_anomalous_events.count()}")
df_anomalous_events.select(
    "winlog_user_name",
    "readable_time",
    "winlog_event_data_param2"
).orderBy(
    "winlog_user_name",
    "readable_time"
).show(truncate=False, n=5000)

## Metrics & Evaluation

**Note:** Ground truth labeling is environment-specific. Update the test configuration below to match your dataset.

**Metrics Strategy:**
- User-week level classification
- Anomaly cluster predicted as POSITIVE
- Normal clusters predicted as NEGATIVE

In [ ]:
# Cell 13: Metrics Helper Function
# =========================================

def calculate_metrics(tp, fp, fn, tn, total_events=None):
    """
    Calculate classification metrics with standard output format.

    Args:
        tp: True Positives
        fp: False Positives
        fn: False Negatives
        tn: True Negatives
        total_events: Optional total for report

    Returns:
        dict with all metrics or None if error
    """
    try:
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f_measure = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = fp / (fp + tp) if (fp + tp) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        print("\n" + "=" * 40)
        print("CLASSIFICATION METRICS REPORT")
        print("=" * 40)
        if total_events:
            print(f"Total Events Evaluated: {total_events}")
        print(f"True Positives (TP):  {tp:,.0f}")
        print(f"False Positives (FP): {fp:,.0f}")
        print(f"False Negatives (FN): {fn:,.0f}")
        print(f"True Negatives (TN):  {tn:,.0f}")
        print("-" * 40)
        print(f"Accuracy:      {accuracy:.4f}")
        print(f"Precision:     {precision:.4f}")
        print(f"Recall:        {recall:.4f}")
        print(f"F1-Measure:    {f_measure:.4f}")
        print(f"False Pos Rate: {fpr:.4f}")
        print(f"False Neg Rate: {fnr:.4f}")
        print("=" * 40)

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f_measure': f_measure,
            'fpr': fpr,
            'fnr': fnr
        }
    except ZeroDivisionError as e:
        print(f"❌ ERROR: Division by zero in metrics calculation: {e}")
        return None

print("✅ Metrics helper function defined")

In [ ]:
# Cell 14: Calculate Confusion Matrix & Metrics
# ====================================================

cluster_sizes = df_predictions.groupBy("prediction").count().orderBy("count")
smallest_two = cluster_sizes.orderBy("count").limit(2)
combined_count = smallest_two.agg(F.sum("count")).collect()[0][0]

test_user = os.getenv('AD_TEST_USER', 'kikki')
test_week = int(os.getenv('AD_TEST_WEEK', '1'))

df_with_labels = df_predictions.withColumn(
    "label",
    F.when(
        (F.col("winlog_user_name") == test_user) & (F.col("week_id") == test_week),
        "MALICIOUS"
    ).otherwise(None)
)

total_events = df_predictions.count()

print(f"\nTotal Events Analyzed (N): {total_events}")
print(f"Test configuration: user={test_user}, week={test_week}")

tp = 0
fn = 0
tn = 0

labels = df_with_labels.select("label").collect()

for row in labels:
    label = row.label

    if label is None:
        tn += 1

    if label == 'MALICIOUS':
        tp += 1

fp = combined_count - tp
tn = tn - fp

print(f"\nConfusion Matrix:")
print(f"TP: {tp} | FP: {fp}")
print(f"FN: {fn} | TN: {tn}")

metrics = calculate_metrics(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    total_events=total_events
)

In [ ]:
# Cell 15: Visualization Helper Function
# =========================================

def plot_confusion_matrix(tp, fp, fn, tn, title='Confusion Matrix'):
    """
    Plot confusion matrix with consistent styling across all notebooks.

    Args:
        tp, fp, fn, tn: Confusion matrix values
        title: Chart title
    """
    import seaborn as sns
    import matplotlib.pyplot as plt

    matrix_values = [[tp, fn], [fp, tn]]

    plt.figure(figsize=(10, 7))

    sns.heatmap(
        matrix_values,
        annot=True,
        fmt=',.0f',
        cmap='Purples',
        linewidths=2,
        linecolor='white',
        xticklabels=['Predicted Positive', 'Predicted Negative'],
        yticklabels=['Actual Positive', 'Actual Negative'],
        cbar=False,
        annot_kws={"size": 16, "weight": "bold"}
    )

    plt.title(title, fontsize=18, fontweight='bold', pad=20)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('Actual Label', fontsize=14)
    plt.tick_params(labelsize=12)
    plt.tight_layout()
    plt.show()

print("✅ Visualization helper function defined")

In [ ]:
# Cell 16: Plot Confusion Matrix
# =========================================

plot_confusion_matrix(
    tp=tp,
    fp=fp,
    fn=fn,
    tn=tn,
    title='Confusion Matrix - AD Enumeration K-Means'
)

## Conclusion

This notebook successfully:
- Configured Spark with environment variables
- Loaded and filtered Directory Services logs
- Engineered TF-IDF features from LDAP query patterns
- Applied K-Means clustering to identify anomalies
- Flagged persistent threats (users in anomaly cluster multiple weeks)
- Provided explainability through cluster center analysis
- Generated classification metrics and confusion matrix

**Next Steps:**
- Investigate users flagged with high anomalous_week_count
- Review top queries defining the anomaly cluster
- Tune hyperparameters (k, num_features) based on dataset size
- Update test user/week configuration for your environment
- Consider implementing automated alerts for persistent threats

**Configuration Changes:**
- Modify `.env` file or set environment variables for:
  - Spark cluster settings (master, memory, cores)
  - MinIO credentials and endpoint
  - Data paths
  - Model hyperparameters (k, num_features, week_size)
  - Test configuration (user, week)